# 时间感知知识图谱智能体实践指南
---

## 模块概述

本笔记本实现了一个时间感知知识图谱智能体系统，用于从非结构化文本中提取实体和关系，并构建具有时间维度的知识图谱。该系统能够处理信息的时效性，支持基于时间的查询和推理，适用于需要跟踪事实变化的场景。

### 主要功能
- 从文本中提取时间感知的三元组（实体-关系-对象）
- 处理不同类型的时间信息（静态、动态、非时间性）
- 管理知识图谱中的事实失效和更新
- 支持多步知识图谱检索和复杂问题回答
- 提供原型到生产的扩展指南

### 设计特点
- 采用三阶段处理管道（时间分类、事件提取、有效性检查）
- 实现双向事实失效检查机制
- 支持多种时间类型（静态、动态、非时间性）和陈述类型（事实、观点、预测）
- 提供灵活的部署和扩展选项

## 目录

1. [执行摘要](#执行摘要)
   1.1. [目的与受众](#目的与受众)
   1.2. [关键要点](#关键要点)

2. [如何使用本指南](#如何使用本指南)
   2.1. [先决条件](#先决条件)

3. [使用时间智能体创建时间感知知识图谱](#使用时间智能体创建时间感知知识图谱)
   3.1. [时间智能体介绍](#时间智能体介绍)
   3.2. [构建时间智能体管道](#构建时间智能体管道)
   3.3. [实体解析与规范化](#实体解析与规范化)
   3.4. [构建知识图谱](#构建知识图谱)

4. [知识图谱的多步检索](#知识图谱的多步检索)
   4.1. [基础知识图谱检索](#基础知识图谱检索)
   4.2. [高级推理与检索策略](#高级推理与检索策略)
   4.3. [检索性能优化](#检索性能优化)

5. [从原型到生产](#从原型到生产)
   5.1. [系统架构扩展](#系统架构扩展)
   5.2. [性能优化](#性能优化)
   5.3. [监控与维护](#监控与维护)

# 1. 执行摘要
---

## 1.1. 目的与受众

本文档旨在指导开发人员构建具有时间感知能力的知识图谱系统，以及如何利用这些系统进行复杂的多步检索和推理。我们将展示如何创建能够跟踪信息随时间变化的智能体，并利用这些智能体构建可查询的知识图谱。

本指南适用于以下人群：
- 数据科学家和机器学习工程师
- 知识图谱和语义网专家
- 需要构建时间感知检索系统的软件开发人员
- 对人工智能和自然语言处理感兴趣的技术决策者

## 1.2. 关键要点

### 时间知识图谱创建

- **时间维度的重要性**：在许多领域（金融、医疗、法律等），信息的时效性对决策至关重要
- **时间智能体架构**：实现三阶段处理管道（时间分类、事件提取、有效性检查）
- **事实失效机制**：通过双向比较和陈述类型约束，高效管理知识更新

### 多步检索

- **结构化查询**：使用图查询语言和推理能力进行复杂检索
- **多跳推理**：支持跨多个关系的知识推理和查询
- **时间敏感回答**：根据特定时间点提供准确的信息检索结果

### 原型到生产

- **可扩展架构**：从实验性原型扩展到生产级系统的指南
- **性能优化**：批处理、缓存和并行化策略
- **健壮性保障**：输出验证、日志记录和监控机制

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>并行化摄取管道</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      从线性文档→分块→提取→解析管道转变为分阶段、异步架构。为每个处理阶段分配自己的队列和专用工作池。对失效作业应用聚类或基于网络的批处理以最大化效率。尽可能批处理外部API请求（例如，OpenAI）和数据库写入。这种设计提高了吞吐量，引入了可靠性背压，并允许您独立扩展每个管道阶段。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>集成稳健的生产保障</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      实施严格的输出验证：标准化时间字段（例如，ISO-8601日期格式），将实体类型限制为受控词汇表，并应用基于模型的轻量级健全性检查以确保输出一致性。使用可追踪标识符进行结构化日志记录，并实时监控质量和性能指标，以便在数据漂移、回归或管道异常影响下游应用之前主动检测它们。
    </p>
  </li>
</ol>

## 2.1. 先决条件

在深入构建时间智能体和知识图谱之前，让我们设置您的环境。使用pip安装所有必需的依赖项，并将OpenAI API密钥设置为环境变量。需要Python 3.12或更高版本。

In [10]:
!python -V
%pip install --upgrade pip
%pip install -qU chonkie datetime ipykernel jinja2 matplotlib networkx numpy openai plotly pydantic rapidfuzz scipy tenacity tiktoken pandas
%pip install -q "datasets<3.0"

Python 3.12.8
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key here: ")

# 3. 使用时间智能体创建时间感知知识图谱
---

**准确的数据是任何良好业务决策的基础。**
OpenAI的最新模型如o3、o4-mini和GPT-4.1系列使企业能够为其最重要的工作流程构建最先进的检索系统。然而，信息快速发展：昨天自信摄取的事实今天可能已经过时。

<!-- ![Benefits of Temporal Knowledge Base](images/01_benefit_of_temporal_kb.jpg) -->
<img
  src="../../../images/01_benefit_of_temporal_kb.jpg"
  alt="时间知识库的好处"
  width="791"
  style="height:auto;"
/>

没有跟踪每个事实何时有效的能力，检索系统可能会返回过时、不合规或误导性的答案。在任何行业中，缺少时间上下文的后果都可能很严重，如以下示例所示。

<table>
  <thead>
    <tr>
      <th>行业</th>
      <th>示例问题</th>
      <th>如果数据库不是时间性的风险</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="3"><strong>金融服务</strong></td>
      <td><em>"自2023年2月以来，穆迪对YY银行的长期评级如何演变？"</em></td>
      <td>通过混合历史和当前评级错误定价信用风险</td>
    </tr>
    <tr>
      <td><em>"ZZ零售商发布2022财年指引时，谁是其CFO？"</em></td>
      <td>治理/内幕交易分析可能会指责错误的高管</td>
    </tr>
    <tr>
      <td><em>"在2024年1月购买CC股票时，AA基金是否受到BB条款的制裁？"</em></td>
      <td>如果规则后来改变，合规报告可能会遗漏违规行为</td>
    </tr>
    <tr>
      <td rowspan="3"><strong>制造业/汽车</strong></td>
      <td><em>"2022年5月至2023年3月期间出厂的Q3型号汽车中部署了哪种ECU固件？"</em></td>
      <td>由于固件漂移而错误诊断现场故障</td>
    </tr>
    <tr>
      <td><em>"8421批次期间，7号装配线上运行的是哪个机器人控制器软件版本？"</em></td>
      <td>根本原因分析可能会指责错误的软件版本</td>
    </tr>
    <tr>
      <td><em>"2024年5月生产的产品中，转向柱螺栓适用什么扭矩规格？"</em></td>
      <td>安全召回可能会遗漏受影响的车辆</td>
    </tr>
  </tbody>
</table>


虽然我们在这里列出了一些具体示例，但这一主题在制药、法律、消费品等许多行业都是适用的。

**超越标准检索**

时间感知知识图谱使您能够超越静态事实查找。它支持更丰富的检索工作流程，如基于时间的事实问答、时间线生成、变更跟踪、反事实分析等。我们将在本指南后面的检索部分更详细地探讨这些内容。

<!-- ![Question types suitable for temporal knowledge bases](./images/02_question_types_for_temporal_kbs.jpg) -->
<img
  src="../../../images/02_question_types_for_temporal_kbs.jpg"
  alt="适用于时间知识库的问题类型"
  style="width:1091px; height:auto;"
/>

## 3.1. 时间智能体介绍
---

**时间智能体**是一个专门的管道，它将原始的自由形式陈述转换为具有时间感知的三元组，准备被摄入到知识图谱中，然后可以用*"在时间T时什么是真实的？"*这类问题进行查询。

三元组是知识图谱的基本构建块。它是一种使用三个部分（因此称为*"三元组"*）表示单个事实或知识片段的方法：
- **主体** - 您正在谈论的实体
- **谓词** - 关系或属性的类型
- **对象** - 主体连接到的值或其他实体

您可以将其视为具有`[主体] - [谓词] - [对象]`结构的句子。更明确的例子：
```
"伦敦" - "isCapitalOf" - "英国"
```

本指南中实现的时间智能体从[Zep](https://arxiv.org/abs/2501.13956)和[Graphiti](https://github.com/getzep/graphiti)中汲取灵感，同时引入了对事实失效的更严格控制和对情节类型的更细致处理方法。

### 3.1.1. 本指南引入的关键增强

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>时间有效性提取</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      基于Graphiti的提示设计，无需辅助参考陈述即可识别时间跨度和情节上下文。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>事实失效逻辑</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      引入双向性检查并通过情节类型约束比较。这保留了Zep的无损方法，同时减少了不必要的评估。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>时间和情节类型</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      区分<code>事实</code>、<code>观点</code>、<code>预测</code>，以及区分时间类别<code>静态</code>、<code>动态</code>、<code>非时间性</code>。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>多事件提取</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      在单次传递中处理复合句和嵌套日期引用。
    </p>
  </li>
</ol>



这个过程使我们能够高效可靠地更新我们的真相来源：

<br>

<!-- ![Statement Invalidation in practice](./images/03_statement_invalidation.png) -->
<img
  src="../../../images/03_statement_invalidation.png"
  alt="实际中的陈述失效"
  style="width:791px; height:auto;"
/>

> **注意**：虽然本指南中的实现专注于基于图的实现，但这种方法可以推广到其他知识库结构，例如基于pgvector的系统。
---

### 3.1.2. 时间智能体管道

时间智能体通过三阶段管道处理传入的陈述：

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>时间分类</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      将每个陈述标记为<strong>非时间性</strong>、<strong>静态</strong>或<strong>动态</strong>：
    </p>
    <ul style="margin-top: 0.5em; margin-bottom: 0.5em; padding-left: 1em;">
      <li style="margin-bottom: 0.5em;"><em>非时间性</em>陈述永不改变（例如，"真空中的光速≈3×10⁸ m s⁻¹"）。</li>
      <li style="margin-bottom: 0.5em;"><em>静态</em>陈述从某个时间点开始有效但之后不会改变（例如，"YY先生于2014年10月23日担任XX公司的CEO"）。</li>
      <li><em>动态</em>陈述会演变（例如，"YY先生是XX公司的CEO"）。</li>
    </ul>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>时间事件提取</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      识别相对或部分日期（例如，"星期二"、"三个月前"）并使用文档时间戳或回退启发式方法将它们解析为绝对日期（例如，如果只知道月份，则默认为该月的第一天或最后一天）。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>时间有效性检查</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      确保每个陈述都包含<code>t_created</code>时间戳，适当时还包含<code>t_expired</code>时间戳。然后，智能体将候选三元组与现有知识图谱条目进行比较，以：
    </p>